# 1c Build your own cache (optional)

**Optional, self-study** — like `08_extra_autograd.ipynb` in spirit: not part of the graded
path, and every other notebook in this lesson works without it. Every cache used so far was
downloaded, per [01_the_cache.ipynb](01_the_cache.ipynb)'s whole argument: encoding is the
part you buy. This notebook is what buying it looks like, kept deliberately tiny so it
finishes on any hardware — a laptop from 2015 included.

If you don't have the hardware, or don't want to spend the time: skip this file entirely.
Nothing downstream depends on it.

In [ ]:
import sys
import time
from pathlib import Path

import datasets
import torch
from datasets import load_dataset
from loguru import logger
from vectormesh import ImageVectorizer, VectorCache

from notebooktester import param

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

torch.manual_seed(0)

# Small on purpose -- see 01_the_cache.ipynb §1.4's measurement: ~80 img/s on a fast CPU,
# 10-30 img/s on an older one. At 100 images that is 1-10 seconds either way.
N_IMAGES = param(100, test=16)
CACHE_DIR = Path("own_caches").resolve()

## Prepare a small slice of a source dataset

Same rule `scripts/build_cache.py` follows for the real, publishable builds: **shuffle
before selecting.** Several hub datasets are sorted by label, so a plain
`.select(range(n))` silently hands back a handful of classes instead of a smaller sample
of the whole task — exactly the DTD-cache trap notebook 05 walks through. `source_idx` is
attached first, so it survives the shuffle and still points back at the row's position in
the *original* split.

In [ ]:
def prepare(source: str, split: str, n: int, seed: int = 0):
    full = load_dataset(source, split=split)
    full = full.add_column("source_idx", list(range(len(full))))
    return full.shuffle(seed=seed).select(range(min(n, len(full))))


SOURCE = "tanganke/eurosat"   # swap this for E2 -- any HF image dataset with train/test splits


def build_split(split: str, n: int):
    return prepare(SOURCE, split, n)

## Build it -- or load it, if you already did

`VectorCache.get_or_create` (new in vectormesh 0.6.0, promoted out of lesson 5's
`cache_utils.py`) is `create`'s idempotent sibling: it checks `CACHE_DIR` for a matching
cache first, and only encodes if nothing is there. Run the next cell twice — the first call
pays the encoding cost, the second is a glob and a disk read.

In [ ]:
vectorizer = ImageVectorizer(model_name="facebook/dinov2-small")
print(f"{vectorizer.model_name} -> dim {vectorizer.get_hidden_size}, device {vectorizer.device}")

start = time.time()
train_cache = VectorCache.get_or_create(
    CACHE_DIR,
    dataset_tag="mini_eurosat_train",
    vectorizer=vectorizer,
    dataset=lambda: build_split("train", N_IMAGES),   # only called on a cache miss
    remove_columns=["image"],
)
print(f"\n{time.time() - start:.1f}s  -- run this cell again and compare")

In [ ]:
test_cache = VectorCache.get_or_create(
    CACHE_DIR,
    dataset_tag="mini_eurosat_test",
    vectorizer=vectorizer,
    dataset=lambda: build_split("test", max(N_IMAGES // 4, 8)),
    remove_columns=["image"],
)

print(f"train {len(train_cache)} rows, test {len(test_cache)} rows")
print(f"dim {train_cache.dim}, {train_cache.num_classes} classes")

## A cheap sanity check

The same 1-nearest-neighbour check [01_the_cache.ipynb](01_the_cache.ipynb) §1.1 opened
with — no training, just distances. At this size (a handful of images per class) it will
not be a good number, but it should be well above chance, which is the actual check: did
the cache come out sane, not "is this a good model".

In [ ]:
Xtr, ytr = train_cache["embed"][:], train_cache["label"][:]
Xte, yte = test_cache["embed"][:], test_cache["label"][:]

A = torch.nn.functional.normalize(Xtr, dim=1)
B = torch.nn.functional.normalize(Xte, dim=1)
neighbour = (B @ A.T).argmax(dim=1)
accuracy = (ytr[neighbour] == yte).float().mean()

print(f"1-NN accuracy: {accuracy:.4f}   chance level: {1 / train_cache.num_classes:.4f}")

## When 100 images is not enough

This notebook exists to make the mechanism felt, not to build a cache anyone should train
on. For a real, publishable build — the same scale the caches you have been downloading
were built at — use `scripts/build_cache.py` directly:

```bash
uv run python scripts/build_cache.py \
    --source tanganke/eurosat --encoder facebook/dinov2-small --tag eurosat
```

or `make build-lesson1` for the full instructor build across every dataset and encoder
this lesson uses. Both need real hardware and real time — §1.4's stopwatch numbers, not
this notebook's.

## Exercises

**E1 — trigger the mismatch guard.** Rerun the train-cache cell above with a *different*
`vectorizer` (say `microsoft/resnet-18`) but the *same* `dataset_tag`
(`"mini_eurosat_train"`). `get_or_create` should raise rather than silently hand you the
DINOv2 cache under a resnet label. Read the error message; it names exactly what
disagreed.

**E2 — build a cache for a model nobody has published.** Pick a different encoder
(anything on the hub `AutoModel` can load) or a different image dataset entirely — an LLM
with hub search is a fast way to shortlist candidates, but the choice of *why* this
encoder for this data should be yours. Change `SOURCE` and `vectorizer` above, give it a
new `dataset_tag`, and rerun. Then run the 1-NN sanity check on it.